# 09. Kesimpulan & Rekomendasi Ilmiah
## Evaluasi Komprehensif Proyek Klasifikasi LULC Jawa Timur dengan SVM (Sentinel-2A)

Notebook ini menyintesis seluruh temuan analitis, capaian metrik model, keberhasilan metodologi mitigasi spasial, serta rekomendasi operasional bagi pemangku kepentingan.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / "output"

print("Direktori output:", OUT_DIR.resolve())

Direktori output: D:\tugas psd\website-jupyterbook\UTS\output


### 1. Sintesis Capaian Kinerja Model SVM
Berdasarkan eksperimen terstandarisasi menggunakan Pipeline `StandardScaler` + `SVC(kernel='rbf', C=10)` pada data uji independen 72 poligon:
- **Akurasi Data Uji**: **88.89%** (64 dari 72 poligon terklasifikasi dengan benar).
- **Macro F1-Score**: **0.8879** (membuktikan performa seimbang pada seluruh 6 kelas tutupan lahan).
- **Weighted F1-Score**: **0.8887**.
- **Kelas Sempurna**: Kelas **Laut** dan **Danau** mencapai *Precision* dan *Recall* mendekati 100% berkat kontras nilai indeks air (NDWI dan MNDWI) yang sangat kuat terhadap objek daratan.

In [2]:
pred_file = OUT_DIR / "df_poligon_pred_svm.csv"
if pred_file.exists():
    df_pred = pd.read_csv(pred_file)
    df_test = df_pred[df_pred['split'] == 'Test']
    rekap_eval = df_test.groupby('label_teks')['benar'].agg(['count', 'sum'])
    rekap_eval['akurasi_%'] = (rekap_eval['sum'] / rekap_eval['count'] * 100).round(2)
    rekap_eval.columns = ['Total Poligon Uji', 'Benar', 'Akurasi (%)']
    print("=== REKAPITULASI KINERJA PER KELAS PADA DATA UJI ===")
    display(rekap_eval)
else:
    print("File prediksi belum tersedia.")

=== REKAPITULASI KINERJA PER KELAS PADA DATA UJI ===


,Total Poligon Uji,Benar,Akurasi (%)
label_teks,,,
Bangunan,12,12,100.00
Danau,12,12,100.00
Hutan,12,8,66.67
Laut,12,12,100.00
Mangrove,12,10,83.33
Sawah,12,10,83.33


### 2. Validitas Metodologi Spasial
1. **Mitigasi Autokorelasi Spasial**: Agregasi centroid per poligon terbukti berhasil menghilangkan akurasi semu (>99%) yang biasa terjadi jika pembagian data dilakukan secara acak pada level piksel. Pembagian data pada tingkat poligon menjamin bahwa poligon uji benar-benar belum pernah dilihat (*unseen*) oleh model.
2. **Representasi Fitur Komprehensif**: Kombinasi 5 band spektral (B02, B03, B04, B08, B11) bersama 4 indeks spektral (NDVI, NDWI, MNDWI, NDBI) memberikan kemampuan pemisahan kelas yang sangat efektif antara vegetasi lebat (Hutan/Mangrove), perairan (Danau/Laut), dan lahan binaan/pertanian (Bangunan/Sawah).

### 3. Temuan pada Prediksi Skala Penuh Raster Jawa Timur
1. **Keberhasilan Inferensi Skala Regional**: Model SVM berhasil memprediksi seluruh piksel valid Jawa Timur beresolusi ~250 m tanpa batasan memori melalui teknik pemrosesan batch 200.000 piksel.
2. **Keterbatasan Mixed Pixel**: Karena resolusi raster ~250 m jauh lebih kasar dibandingkan data sampel latih (10–20 m), terdapat fenomena piksel campuran pada garis pantai dan sempadan objek sempit.
3. **Rekomendasi Operasional**: Disarankan menerapkan komposit temporal multi-musim (kemarau dan hujan) untuk mempertajam pemisahan antara sawah produktif dan sawah bero/kering, serta menerapkan *water-masking* sebelum klasifikasi daratan.

In [3]:
luas_file = OUT_DIR / "07_luas_per_kelas.csv"
if luas_file.exists():
    df_luas = pd.read_csv(luas_file)
    print("=== ESTIMASI LUASAN TUTUPAN LAHAN REGIONAL JAWA TIMUR ===")
    display(df_luas)
else:
    print("File estimasi luas belum tersedia.")

=== ESTIMASI LUASAN TUTUPAN LAHAN REGIONAL JAWA TIMUR ===


,kelas_id,kelas_teks,jumlah_piksel,luas_km2,persentase
0,1,Sawah,203169,15487.40,12.71
1,2,Bangunan,64904,4947.58,4.06
2,3,Hutan,409671,31228.87,25.62
3,4,Danau,124949,9524.76,7.82
4,5,Laut,759193,57872.64,47.49
5,6,Mangrove,36841,2808.36,2.30
